# Weaviate To Kayak Reranking

This notebook shows a complete local path:

1. start an embedded Weaviate instance
2. create a self-provided multivector collection
3. insert ColBERT-style token matrices as a named vector
4. filter candidates in Weaviate
5. fetch candidate vectors with `include_vector=["colbert"]`
6. rerank those candidates exactly in Kayak with the Mojo backend
7. inspect clause-text verification
8. measure local average timings

Install for this notebook with:

```bash
uv add weaviate-client
```

or:

```bash
pixi add --pypi weaviate-client
```

In [1]:
import time
import warnings

import numpy as np
import kayak
from colbert.infra.config import ColBERTConfig
from colbert.modeling.checkpoint import Checkpoint
import torch

DEFAULT_MODEL_NAME = "colbert-ir/colbertv2.0"

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="CUDA is not available.*", category=UserWarning)
warnings.filterwarnings(
    "ignore",
    message="torch.cuda.amp.GradScaler is enabled, but CUDA is not available.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message="resource_tracker: There appear to be .* leaked semaphore objects.*",
    category=UserWarning,
)

torch.set_num_threads(1)
torch.multiprocessing.set_sharing_strategy("file_system")

checkpoint = Checkpoint(
    DEFAULT_MODEL_NAME,
    colbert_config=ColBERTConfig(gpus=0),
    verbose=0,
)

def encode_query_text(text: str) -> list[list[float]]:
    with torch.inference_mode():
        encoded = checkpoint.queryFromText([text], to_cpu=True)
    return encoded[0].detach().cpu().tolist()


def encode_document_text(text: str) -> list[list[float]]:
    with torch.inference_mode():
        encoded = checkpoint.docFromText([text], to_cpu=True)
    return encoded[0].detach().cpu().tolist()

warnings.filterwarnings("ignore", category=DeprecationWarning)

BACKEND = kayak.MOJO_EXACT_CPU_BACKEND

def timed_avg_ms(fn, repeats=50, warmup=5):
    for _ in range(warmup):
        fn()
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    return (time.perf_counter() - start) * 1000.0 / repeats

def describe_hits(hits):
    rows = []
    for hit in hits:
        row = {
            "doc_id": hit.doc_id,
            "score": round(float(hit.score), 3),
        }
        clause_text = getattr(hit, "clause_text", None)
        if clause_text:
            row["text"] = clause_text
        rows.append(row)
    return rows

DOCS = [
    {
        "doc_id": "pixi_mojo_project",
        "topic": "installation",
        "text": "Create one Pixi project, install Python, Mojo, and kayak in the same environment, then set BACKEND = kayak.MOJO_EXACT_CPU_BACKEND in your application code.",
    },
    {
        "doc_id": "pixi_python_pin",
        "topic": "installation",
        "text": "Pixi can pin Python 3.11 and add PyPI packages together with Mojo in a single project environment.",
    },
    {
        "doc_id": "uv_kayak_only",
        "topic": "installation",
        "text": "uv add kayak installs the Python package but does not install the Mojo CLI, so the NumPy reference backend remains the only available backend.",
    },
    {
        "doc_id": "qdrant_multivector",
        "topic": "vector_db",
        "text": "Qdrant can store ColBERT-style multivectors and search them with a MAX_SIM comparator in one collection.",
    },
    {
        "doc_id": "weaviate_multivector",
        "topic": "vector_db",
        "text": "Weaviate embedded mode can store self-provided multivectors under a named vector such as colbert.",
    },
    {
        "doc_id": "lancedb_multivector",
        "topic": "vector_db",
        "text": "LanceDB can store multivector columns as list-of-list float data for local reranking experiments.",
    },
    {
        "doc_id": "clause_text_verifier",
        "topic": "search_plan",
        "text": "Kayak clause_text stage 3 verification returns supporting text for the final hits.",
    },
    {
        "doc_id": "stage_profiles",
        "topic": "search_plan",
        "text": "Kayak search plans expose candidate counts, stage 2 work, and verifier materialization counts explicitly.",
    },
]

query_text = "How should a project install Mojo and make Kayak use the Mojo backend by default?"
query_vectors = np.asarray(encode_query_text(query_text), dtype=np.float32)
query = kayak.query(query_vectors, text=query_text)

doc_ids = [doc["doc_id"] for doc in DOCS]
doc_texts = [doc["text"] for doc in DOCS]
doc_vectors = [np.asarray(encode_document_text(doc["text"]), dtype=np.float32) for doc in DOCS]

full_index = kayak.documents(doc_ids, doc_vectors, texts=doc_texts).pack()

{
    "query_vector_count": int(query_vectors.shape[0]),
    "document_vector_counts": {
        doc_id: int(vectors.shape[0])
        for doc_id, vectors in zip(doc_ids, doc_vectors)
    },
}

/tmp/kayak-vdb-tools/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Apr 15, 12:46:35] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


{'query_vector_count': 32,
 'document_vector_counts': {'pixi_mojo_project': 43,
  'pixi_python_pin': 26,
  'uv_kayak_only': 36,
  'qdrant_multivector': 29,
  'weaviate_multivector': 24,
  'lancedb_multivector': 26,
  'clause_text_verifier': 19,
  'stage_profiles': 25}}

## Start Embedded Weaviate And Insert Multivectors

In [2]:
import shutil

import weaviate
import weaviate.classes as wvc

persistence_path = '/tmp/weaviate-kayak-docs'
shutil.rmtree(persistence_path, ignore_errors=True)

client = weaviate.connect_to_embedded(
    persistence_data_path=persistence_path,
    environment_variables={'LOG_LEVEL': 'error'},
)

if client.collections.exists('Doc'):
    client.collections.delete('Doc')

collection = client.collections.create(
    name='Doc',
    properties=[
        wvc.config.Property(name='doc_id', data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name='topic', data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name='text', data_type=wvc.config.DataType.TEXT),
    ],
    vector_config=wvc.config.Configure.MultiVectors.self_provided(name='colbert'),
)

for doc, vectors in zip(DOCS, doc_vectors):
    collection.data.insert(
        properties={
            'doc_id': doc['doc_id'],
            'topic': doc['topic'],
            'text': doc['text'],
        },
        vector={'colbert': vectors.tolist()},
    )

'ready'

'ready'

## Stage 1 Retrieval In Weaviate

In [3]:
def weaviate_stage1(limit=3):
    return collection.query.near_vector(
        near_vector={'colbert': query_vectors.tolist()},
        target_vector='colbert',
        filters=wvc.query.Filter.by_property('topic').equal('installation'),
        limit=limit,
        include_vector=['colbert'],
        return_properties=['doc_id', 'topic', 'text'],
        return_metadata=wvc.query.MetadataQuery(distance=True),
    ).objects

candidate_objects = weaviate_stage1()
[
    (obj.properties['doc_id'], round(float(obj.metadata.distance), 3))
    for obj in candidate_objects
]

[('pixi_mojo_project', -22.654),
 ('uv_kayak_only', -21.87),
 ('pixi_python_pin', -16.529)]

## Exact Kayak Rerank On Weaviate Candidates

In [4]:
def weaviate_candidate_index(objects):
    return kayak.documents(
        [obj.properties['doc_id'] for obj in objects],
        [np.asarray(obj.vector['colbert'], dtype=np.float32) for obj in objects],
        texts=[obj.properties['text'] for obj in objects],
    ).pack()

candidate_index = weaviate_candidate_index(candidate_objects)
reranked_hits = kayak.search(query, candidate_index, k=2, backend=BACKEND)
describe_hits(reranked_hits)

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type PreparedPackedIndex has no __module__ attribute


[{'doc_id': 'pixi_mojo_project', 'score': 22.654},
 {'doc_id': 'uv_kayak_only', 'score': 21.87}]

## Clause-Text Verification On The Same Candidates

In [5]:
verifier_plan = kayak.exact_full_scan_search_plan(
    final_k=2,
    candidate_k=len(candidate_objects),
    stage3_verifier=kayak.clause_text_stage3_verifier_operator(),
)

verifier_result = kayak.search_with_plan(
    query,
    candidate_index,
    verifier_plan,
    backend=BACKEND,
)

print('stage3_hits:')
print(describe_hits(verifier_result.hits))
print('stage2_profile:', verifier_result.stage2)
print('stage3_profile:', verifier_result.stage3_verifier)

stage3_hits:
[{'doc_id': 'pixi_mojo_project', 'score': 25.654}, {'doc_id': 'uv_kayak_only', 'score': 24.27}]
stage2_profile: SearchStageProfile(stage_name='noop_topk', input_hit_count=3, output_hit_count=3, query_vector_count=0, document_count=3, document_vector_count=0, document_text_count=0, materialized_artifacts=())
stage3_profile: SearchStageProfile(stage_name='clause_text', input_hit_count=3, output_hit_count=2, query_vector_count=0, document_count=3, document_vector_count=0, document_text_count=3, materialized_artifacts=(StageArtifactMaterialization(family='document_text', document_count=3, document_vector_count=0, document_text_count=3),))


## Compare Against A Full Exact Kayak Baseline

In [6]:
exact_hits = kayak.search(query, full_index, k=2, backend=BACKEND)

comparison = {
    'weaviate_candidate_ids': [obj.properties['doc_id'] for obj in candidate_objects],
    'kayak_reranked_ids': [hit.doc_id for hit in reranked_hits],
    'full_exact_ids': [hit.doc_id for hit in exact_hits],
}
comparison

{'weaviate_candidate_ids': ['pixi_mojo_project',
  'uv_kayak_only',
  'pixi_python_pin'],
 'kayak_reranked_ids': ['pixi_mojo_project', 'uv_kayak_only'],
 'full_exact_ids': ['pixi_mojo_project', 'uv_kayak_only']}

## Local Average Timings

In [7]:
def weaviate_plus_kayak():
    objects = weaviate_stage1()
    idx = weaviate_candidate_index(objects)
    return kayak.search(query, idx, k=2, backend=BACKEND)

timings_ms = {
    'weaviate_stage1_ms': timed_avg_ms(weaviate_stage1, repeats=25),
    'weaviate_plus_kayak_ms': timed_avg_ms(weaviate_plus_kayak, repeats=15),
    'full_kayak_exact_ms': timed_avg_ms(
        lambda: kayak.search(query, full_index, k=2, backend=BACKEND),
        repeats=100,
    ),
}

{name: round(value, 3) for name, value in timings_ms.items()}

{'weaviate_stage1_ms': 3.879,
 'weaviate_plus_kayak_ms': 5.784,
 'full_kayak_exact_ms': 0.688}

In [8]:
client.close()

{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","error":"cannot find peer","level":"error","msg":"transferring leadership","time":"2026-04-15T12:46:39+02:00"}
{"action":"load_all_shards","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"error","msg":"failed to load all shards: context canceled","time":"2026-04-15T12:46:39+02:00"}
